In [ ]:
import random
import os
import numpy as np
import time
import torch
import torch.nn as nn 
import pandas as pd 
import argparse
import torch.optim as optim
import torch.utils.data as data
from tensorboardX import SummaryWriter

参数设置：

In [ ]:
#设置伪随机数，方便实验结果复现
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # 即使没有 CUDA，也可以设置，不会产生影响
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # 在没有 CUDA 的情况下通常将其设置为 False

In [ ]:
parser = argparse.ArgumentParser() 
parser.add_argument("--seed", type=int, default=42,help="Seed")
parser.add_argument("--lr", type=float, default=0.001, help="learning rate")
parser.add_argument("--dropout",type=float,default=0.2, help="dropout rate")
parser.add_argument("--batch_size", type=int, default=256,help="batch size for training")
parser.add_argument("--epochs", type=int,default=10,  help="training epoches")
parser.add_argument("--top_k", type=int, default=10, help="compute metrics@top_k")
parser.add_argument("--factor_num", type=int,default=32, help="predictive factors numbers in the model")
parser.add_argument("--layers",nargs='+', default=[64,32,16,8], help="MLP layers. Note that the first layer is the concatenation of user and item embeddings. So layers[0]/2 is the embedding size.")
parser.add_argument("--num_ng", type=int,default=4, help="Number of negative samples for training set")
parser.add_argument("--num_ng_test", type=int,default=100, help="Number of negative samples for test set")
parser.add_argument("--out", default=True,help="save model or not")

args=parser.parse_known_args()[0]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
writer = SummaryWriter()

seed_everything(args.seed)

处理数据：

In [ ]:
"""
这个自定义数据集类的主要目的是将原始数据组织成适用于 PyTorch 模型训练和测试的形式。
通过实现这些方法，可以方便地使用 PyTorch 提供的数据加载器来加载和处理数据，以供深度学习模型使用。
"""
class Rating_Datset(torch.utils.data.Dataset):
    def __init__(self, user_list, item_list, rating_list):
        super(Rating_Datset, self).__init__()
        self.user_list = user_list
        self.item_list = item_list
        self.rating_list = rating_list

    def __len__(self):
        return len(self.user_list)

    def __getitem__(self, idx):
        user = self.user_list[idx]
        item = self.item_list[idx]
        rating = self.rating_list[idx]

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(item, dtype=torch.long),
            torch.tensor(rating, dtype=torch.float)
            )

In [ ]:
class NCF_Data(object):
    """
    Construct Dataset for NCF
    """
    def __init__(self, args, ratings1, ratings2):
        self.ratings1 = ratings1
        self.ratings2 = ratings2
        self.num_ng = args.num_ng
        self.num_ng_test = args.num_ng_test
        self.batch_size = args.batch_size
        
        self.train_ratings = self.ratings1
        self.test_ratings = self.ratings2
        random.seed(args.seed)


    def get_train_instance(self):
        users, items, ratings = [], [], []
        train_ratings = self.train_ratings
        # 将训练评分数据集 self.train_ratings 与负样本数据 self.negatives[['user_id', 'negative_items']] 进行合并，以获得包含正样本和负样本的数据                                              
        for row in train_ratings.itertuples():
            users.append(row.user_id)
            items.append(row.item_id)
            ratings.append(row.rating)
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True, num_workers=0)#num_workers=4改成0，4会发生broken pipe

    def get_test_instance(self):
        users, items, ratings = [], [], []
        test_ratings = self.test_ratings
        for row in test_ratings.itertuples():
            users.append(row.user_id)
            items.append(row.item_id)
            ratings.append(row.rating)
        dataset = Rating_Datset(
            user_list=users,
            item_list=items,
            rating_list=ratings)
        return torch.utils.data.DataLoader(dataset, batch_size=self.num_ng_test+1, shuffle=False, num_workers=0)#num_workers=4改成0

In [ ]:
class Generalized_Matrix_Factorization(nn.Module):
    def __init__(self, args, num_users, num_items):#args包含模型的超参数
        super(Generalized_Matrix_Factorization, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.factor_num = args.factor_num#表示嵌入向量的维度

        self.embedding_user = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num)
        self.embedding_item = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num)

        self.affine_output = nn.Linear(in_features=self.factor_num, out_features=1)#全连接层
        self.logistic = nn.Sigmoid()

    def forward(self, user_indices, item_indices):#前向传播函数
        user_embedding = self.embedding_user(user_indices)
        item_embedding = self.embedding_item(item_indices)
        element_product = torch.mul(user_embedding, item_embedding)
        logits = self.affine_output(element_product)#元素积
        rating = self.logistic(logits)
        return rating

    def init_weight(self):
        pass

class Multi_Layer_Perceptron(nn.Module):
    def __init__(self, args, num_users, num_items):
        super(Multi_Layer_Perceptron, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.factor_num = args.factor_num
        self.layers = args.layers

        self.embedding_user = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num)
        self.embedding_item = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num)

        self.fc_layers = nn.ModuleList()#创建了空列表用于存储多个全连接层
        for idx, (in_size, out_size) in enumerate(zip(self.layers[:-1], self.layers[1:])):#切片的语法
# 这是一个循环，它同时迭代遍历 self.layers[:-1] 和 self.layers[1:] 中的元素对，将它们分配给变量 idx（索引）、in_size（输入维度）和 out_size（输出维度）。
            self.fc_layers.append(nn.Linear(in_size, out_size))
# append 方法将这个全连接层对象添加到 self.fc_layers 中，将其存储起来以备后续使用
        self.affine_output = nn.Linear(in_features=self.layers[-1], out_features=1)
        self.logistic = nn.Sigmoid()

    def forward(self, user_indices, item_indices):
        user_embedding = self.embedding_user(user_indices)
        item_embedding = self.embedding_item(item_indices)
        vector = torch.cat([user_embedding, item_embedding], dim=-1)  # the concat latent vector
        for idx, _ in enumerate(range(len(self.fc_layers))):
            vector = self.fc_layers[idx](vector)
            vector = nn.ReLU()(vector)
            # vector = nn.BatchNorm1d()(vector)
            # vector = nn.Dropout(p=0.5)(vector)
        logits = self.affine_output(vector)
        rating = self.logistic(logits)
        return rating

    def init_weight(self):
        pass

In [ ]:
class NeuMF(nn.Module):
    def __init__(self, args, num_users, num_items):
        super(NeuMF, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.factor_num_mf = args.factor_num#MF部分的潜在因子数量32
        self.factor_num_mlp =  int(args.layers[0]/2)#MLP部分的潜在因子数量32
        self.layers = args.layers #MLP各层大小
        self.dropout = args.dropout #正则化dropout rating

        self.embedding_user_mlp = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num_mlp)
        self.embedding_item_mlp = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num_mlp)

        self.embedding_user_mf = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.factor_num_mf)
        self.embedding_item_mf = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.factor_num_mf)

        self.fc_layers = nn.ModuleList()
        # 由多个全连接层组成的列表，每个全连接层后跟一个ReLU激活函数
        for idx, (in_size, out_size) in enumerate(zip(args.layers[:-1], args.layers[1:])):
            self.fc_layers.append(torch.nn.Linear(in_size, out_size))
            self.fc_layers.append(nn.ReLU())

        self.affine_output = nn.Linear(in_features=args.layers[-1] + self.factor_num_mf, out_features=1)
        self.logistic = nn.Sigmoid()
        self.init_weight()

    """
    init_weight 方法通过不同的初始化策略对模型的权重进行初始化，
    以确保模型在训练过程中能够更好地收敛，并且不会出现梯度问题
    """   
    def init_weight(self):
        nn.init.normal_(self.embedding_user_mlp.weight, std=0.01)
        nn.init.normal_(self.embedding_item_mlp.weight, std=0.01)
        nn.init.normal_(self.embedding_user_mf.weight, std=0.01)
        nn.init.normal_(self.embedding_item_mf.weight, std=0.01)
        
        for m in self.fc_layers:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                
        nn.init.xavier_uniform_(self.affine_output.weight)

        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                m.bias.data.zero_()#将当前子模块的偏差（如果存在）初始化为零

    def forward(self, user_indices, item_indices):
        user_embedding_mlp = self.embedding_user_mlp(user_indices)
        item_embedding_mlp = self.embedding_item_mlp(item_indices)

        user_embedding_mf = self.embedding_user_mf(user_indices)
        item_embedding_mf = self.embedding_item_mf(item_indices)

        mlp_vector = torch.cat([user_embedding_mlp, item_embedding_mlp], dim=-1)  # the concat latent vector
        mf_vector =torch.mul(user_embedding_mf, item_embedding_mf)

        for idx, _ in enumerate(range(len(self.fc_layers))):
        #实现了 MLP 的前向传播过程，它将输入数据 mlp_vector 通过 MLP 中的每一层逐层传递，最终得到 MLP 的输出
            mlp_vector = self.fc_layers[idx](mlp_vector)

        vector = torch.cat([mlp_vector, mf_vector], dim=-1)
        logits = self.affine_output(vector)
        rating = self.logistic(logits)
        return rating.squeeze()
        #return rating

评估：

In [ ]:
def hit(ng_item, pred_items):
    if ng_item in pred_items:
        return 1
    return 0

def ndcg(ng_item, pred_items):
    if ng_item in pred_items:
        index = pred_items.index(ng_item)
        return np.reciprocal(np.log2(index+2))
    return 0

def metrics(model, test_loader, top_k, device):
    HR, NDCG = [], []

    for user, item, label in test_loader:
        user = user.to(device)
        item = item.to(device)

        predictions = model(user, item)#不同用户对不同物品的喜好程度0-1
        _, indices = torch.topk(predictions, top_k)
        recommends = torch.take(
                item, indices).cpu().numpy().tolist()

        ng_item = item[0].item() # leave one-out evaluation has only one item per user
        HR.append(hit(ng_item, recommends))
        NDCG.append(ndcg(ng_item, recommends))
    return np.mean(HR), np.mean(NDCG)

In [ ]:
n=5
MODEL_PATH = r'/JupyterCode/NCF/'

for part in range(1,n+1):
    MODEL = f'./models/model_div{n}/div_shard{part}.pth'
    traindataset = pd.read_csv(
        f'./data/div_shard{n}/sub_train{part}.csv',
        sep=",", 
        names = ['user_id', 'item_id', 'rating'], 
        engine='python')
    testdataset = pd.read_csv(
        './data/data_div_train_and_test/test_data.csv',
        sep=",", 
        names = ['user_id', 'item_id', 'rating'], 
        engine='python')
    # construct the train and test datasets
    data = NCF_Data(args, traindataset,testdataset)
    train_loader =data.get_train_instance()
    test_loader =data.get_test_instance()

    loss_function = nn.BCELoss()

    # set the num_users, items
    num_users = 82536
    num_items = traindataset['item_id'].nunique()+1

    # set model and optimizer
    model =NeuMF(args, num_users, num_items)
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    # train, evaluation
    best_hr = 0
    start_time = time.time()
    for epoch in range(1, args.epochs+1):
        model.train() # Enable dropout (if have).

        for user, item, label in train_loader:
            user = user.to(device)
            item = item.to(device)
            label = label.to(device)

            optimizer.zero_grad()#清零梯度
            prediction = model(user, item)   
            loss = loss_function(prediction, label)
            loss.backward()
            optimizer.step()
            writer.add_scalar('loss/Train_loss', loss.item(), epoch)

        model.eval()
        HR, NDCG = metrics(model, test_loader, args.top_k, device)
        writer.add_scalar('Perfomance/HR@10', HR, epoch)
        writer.add_scalar('Perfomance/NDCG@10', NDCG, epoch)
        print("HR: {:.3f}\tNDCG: {:.3f}".format(np.mean(HR), np.mean(NDCG)))

        if HR > best_hr:
            elapsed_time = time.time() - start_time#到最好HR的epoch花费的时间
            best_hr, best_ndcg, best_epoch = HR, NDCG, epoch
            if args.out:
                if not os.path.exists(MODEL_PATH):#删去了config.
                    os.mkdir(MODEL_PATH)
                torch.save(model.state_dict(), 
                    '{}'.format(MODEL))
                
    writer.close()
    print("The time elapse of best epoch is: " + 
                    time.strftime("%H: %M: %S", time.gmtime(elapsed_time)))
    print("End. Best epoch of part{:03d}: HR = {:.3f}, NDCG = {:.3f}".format(part, best_hr, best_ndcg))
    
